# Feature Engineering with information available at renewal

## Overview

- The goal of this notebook is to engineer features from variables of dataset that have been obtained by an underwriter(s) at least first renewal. This implies that we may now have more internal data available our our disposal.
- This will be tackled in 4 waves in terms of the data now available to us
    - Driver Features
    - Payment Features
    - Claims Features
    - Vehicle Features

## Setup

In [3]:
import os
from datetime import datetime
from typing import Any
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from langchain_core.language_models.chat_models import agenerate_from_stream
from sklearn import preprocessing

os.chdir('..') ## on the assumption that this is rum from notebook/freq_sev

In [4]:
pwd

'/Users/olumide/PycharmProjects/underwriting-assessor'

### Load the data

In [5]:
insurance_initiation_variables_path = "data/input/Motor_vehicle_insurance_data.csv"
claims_variables_path = "data/input/exp/sample_type_claim.csv"
insurance_df = pd.read_csv(insurance_initiation_variables_path, delimiter=';')
claims_df = pd.read_csv(claims_variables_path, delimiter=';')

## 1. Prepare Dataset

| Step | Action |
|------|--------|
| Aggregate | Sum claims by (ID, year) |
| Merge | Left join on (ID, year) |
| Fill | NaN → 0 for no claims |
| Split | 80/20 train-test split |

### 1.1 Aggregate claims by policyholder and year

In [6]:
claims_frequency  = (
    claims_df
    .groupby(['ID', 'Cost_claims_year'])
    .agg({'Cost_claims_by_type': 'count'})
    .rename(columns={'Cost_claims_by_type': 'claims_frequency'})
    .reset_index()
)

### 1.2 Merge insurance and claims data

In [7]:
dataset = (
    pd
    .merge(
        left=insurance_df,
        right=claims_frequency,
        how='left',
        on=['ID', 'Cost_claims_year']
    )
    .fillna(value={'claims_frequency':0})
)

### 1.3 Split into train and test sets

In [ ]:
test_ratio = 0.2
to_shuffle = False
if to_shuffle:
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

split_index = int(len(dataset) * (1 - test_ratio))
trainset = dataset.iloc[:split_index].reset_index(drop=True)
testset = dataset.iloc[split_index:].reset_index(drop=True)
print(f"Total records: {len(dataset)}")
print(f"Training set: {len(trainset)} records ({100*(1-test_ratio):.0f}%)")
print(f"Test set: {len(testset)} records ({100*test_ratio:.0f}%)")

## 2. Feature Engineering

While there are now advancement in the motor insurance sector that enables using telematic data points, I will be exploring traditional features in this segment mostly based on domain knowledge [some capture here](https://www.researchgate.net/publication/338007809_An_Analysis_of_the_Risk_Factors_Determining_Motor_Insurance_Premium_in_a_Small_Island_State_The_Case_of_Malta).


### 2.1 - Driver Features
The below are directly available in the dataset
- Driver DOB
- Driver Driving License date
- Second driver on the car

Some features to engineer from the above
- Respective driver age at inception of each contract
- Evidence backed driver age banding at the inception of each contract
- Length of driving license (driving experience) at the inception of each contract
- Evidence backed license length banding at the inception of each contract.
- Interaction between second driver & driver age banding - as a proxyto identify couples or parent/children?
    - e.g young single driver may be considered high risk
    - older driver with second driver may be low to moderate risk (although we cant conclude with this)
- Age experience gap for each contract. Difference between age and driving experience in years. Lower is better? But how low?
    - So a 25 year old with 1 year experience has agegapexp = 24
    - 25 years old with 6 years experience has agegapexp=19

In [8]:
dataset['Date_last_renewal'] = pd.to_datetime(dataset['Date_last_renewal'], format='%d/%m/%Y', dayfirst=True)
dataset['Date_birth'] = pd.to_datetime(dataset['Date_birth'], format='%d/%m/%Y', dayfirst=True)
dataset['driver_age_at_contract_start'] =  dataset['Date_last_renewal'].dt.year - dataset['Date_birth'].dt.year
dataset['Date_driving_licence'] = pd.to_datetime(dataset['Date_driving_licence'], format='%d/%m/%Y', yearfirst=True)
dataset['driver_experience_age'] = dataset['Date_last_renewal'].dt.year - dataset['Date_driving_licence'].dt.year
dataset['driver_age_experience_age_diff'] = abs(dataset['driver_age_at_contract_start'] - dataset['driver_experience_age'])
dataset['driver_age_experience_ratio_proxy_for_driving_experience'] = dataset['driver_age_experience_age_diff'] / dataset['driver_age_at_contract_start']
dataset[['claims_frequency', 'driver_age_experience_ratio_proxy_for_driving_experience', 'driver_age_at_contract_start', 'driver_age_experience_age_diff']].corr(method='pearson')


,claims_frequency,driver_age_experience_ratio_proxy_for_driving_experience,driver_age_at_contract_start,driver_age_experience_age_diff
claims_frequency,1.000000,0.044412,-0.036871,0.009732
driver_age_experience_ratio_proxy_for_driving_experience,0.044412,1.000000,-0.654786,0.481074
driver_age_at_contract_start,-0.036871,-0.654786,1.000000,0.295723
driver_age_experience_age_diff,0.009732,0.481074,0.295723,1.000000


Having tried all the above features, the engineered features showing the best relationship is driving age experience ratio.
- Calculated by - (difference between drivers age at contract inception and total driving experience in years) / drivers age at contract inception
- This gives a ratio between 0 and 1. Values closer to 1 mean that driver is inexperienced while values closer to zero indicates very experienced drivers
- So for example
    - Driver A with driver age 19years and driving experience 1year. Ratio = (19-1)/19 = 0.95 (inexperienced driver)
    - Driver B with driver age 30years and driving experience 12 years. Ratio = (30-12)/30 = 0.6 (decently experienced driver)

So this feature simply says how experienced is the driver relative to their opportunity to have gained experience

### Payment Features
The below are directly available in the dataset as payment related attributes
- Lapse (number of policy cancelled due to default for a single maturing year)
- Date Lapse (default date)
- Payment (annually=0 or half-year administrative process=1)

Premium excluded for obvious leakage reasons.

Some feature engineering ideas
- Bin the lapse into 0 and 1+ lapse. This will capture customers that have not cancelled for any reason or have had to terminate a contract within the year. A signal for dissatisfaction or financial distress?
- Use payment as it is.



In [9]:
dataset['Payment'].value_counts(normalize=True) * 100

Payment
0    68.082043
1    31.917957
Name: proportion, dtype: float64

In [10]:
(
    dataset[['Payment', 'Cost_claims_year','ID']]
    .groupby('Payment').agg(
    {
        'ID':'count',
        'Cost_claims_year':'mean',
    }
    )
    .rename(columns={'ID':'policy_count'})

)

,policy_count,Cost_claims_year
Payment,,
0,71864,120.123302
1,33691,224.873122


In [11]:
dataset['lapse_flag'] = (dataset['Lapse'] >= 1).astype(int)
dataset[['claims_frequency','lapse_flag', 'Payment']].corr(method='pearson')

,claims_frequency,lapse_flag,Payment
claims_frequency,1.000000,-0.051976,0.044805
lapse_flag,-0.051976,1.000000,0.068941
Payment,0.044805,0.068941,1.000000


- Payment would be making it as a solid feature candidate. Despite half-year payment mode being used for circa 30% of the dataset, the average claims cost for this group is 2x that of the annual payment group.
- Lapse flag - still in the back pocket

### 2.3 - Claims Feature

- N_claims_year - this is the same as claims frequency (for each year). was calculated and just renamed differently in the dataset - to be excluded
- N_claims_history - this will be excluded as well because it is the claims freaquency on the policy which is since inception of that policy
- R_Claims_history - this is total number of claims / duration of policy

In [12]:
x = (dataset[['ID','claims_frequency', 'N_claims_year', 'R_Claims_history', 'N_claims_history']])
x.corr(method='spearman')

,ID,claims_frequency,N_claims_year,R_Claims_history,N_claims_history
ID,1.000000,0.016724,0.016193,0.009502,-0.071536
claims_frequency,0.016724,1.000000,0.447919,0.267851,0.103703
N_claims_year,0.016193,0.447919,1.000000,0.604894,0.376832
R_Claims_history,0.009502,0.267851,0.604894,1.000000,0.654019
N_claims_history,-0.071536,0.103703,0.376832,0.654019,1.000000


- Fill comfortable taking forward R claims history which is the total number of claims for a policy which dates outside of the dataset year cut off / total duration in years of the policy in force

### Vehicle Features

There are few features

- Year of registration
- Vehicle power
- Cylinder capacity
- Value of vehicle
- Number of doors
- Fuel Type
- Length
- Weight

Some feature transformation in mind
1. Log transform the value of the vehicle

In [13]:
dataset['fuel_type_encoded'], fuel_type_uniques = pd.factorize(dataset['Type_fuel'])
dataset[['claims_frequency', 'fuel_type_encoded', 'Value_vehicle', 'N_doors', 'Length', 'Cylinder_capacity', 'Power', ]].corr(method='spearman')

,claims_frequency,fuel_type_encoded,Value_vehicle,N_doors,Length,Cylinder_capacity,Power
claims_frequency,1.000000,0.029268,0.030932,0.020564,0.015729,0.030638,0.031940
fuel_type_encoded,0.029268,1.000000,0.473941,0.283055,0.310847,0.565008,0.306028
Value_vehicle,0.030932,0.473941,1.000000,0.240891,0.766530,0.693616,0.763393
N_doors,0.020564,0.283055,0.240891,1.000000,0.010196,0.171201,0.219939
Length,0.015729,0.310847,0.766530,0.010196,1.000000,0.637831,0.594449
Cylinder_capacity,0.030638,0.565008,0.693616,0.171201,0.637831,1.000000,0.579250
Power,0.031940,0.306028,0.763393,0.219939,0.594449,0.579250,1.000000


In [15]:
dataset['Value_vehicle_log_transfomed'] = dataset['Value_vehicle'].apply(np.log)
dataset

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Length,Weight,claims_frequency,driver_age_at_contract_start,driver_experience_age,driver_age_experience_age_diff,driver_age_experience_ratio_proxy_for_driving_experience,lapse_flag,fuel_type_encoded,Value_vehicle_log_transfomed
0,1,05/11/2015,2015-11-05,05/11/2016,1956-04-15,1976-03-20,0,4,1,2,...,NaN,190,0.0,59,39,20,0.338983,0,0,8.863333
1,1,05/11/2015,2016-11-05,05/11/2017,1956-04-15,1976-03-20,0,4,1,2,...,NaN,190,0.0,60,40,20,0.333333,0,0,8.863333
2,1,05/11/2015,2017-11-05,05/11/2018,1956-04-15,1976-03-20,0,4,2,2,...,NaN,190,0.0,61,41,20,0.327869,0,0,8.863333
3,1,05/11/2015,2018-11-05,05/11/2019,1956-04-15,1976-03-20,0,4,2,2,...,NaN,190,0.0,62,42,20,0.322581,0,0,8.863333
4,2,26/09/2017,2017-09-26,26/09/2018,1956-04-15,1976-03-20,0,4,2,2,...,NaN,190,0.0,61,41,20,0.327869,0,0,8.863333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105550,53498,30/07/2018,2018-07-30,30/07/2019,1981-07-25,2007-02-14,0,1,1,1,...,4.740,1480,0.0,37,11,26,0.702703,0,1,10.099054
105551,53499,16/08/2018,2018-08-16,16/08/2019,1976-12-08,2017-11-29,0,1,1,1,...,4.650,1440,0.0,42,1,41,0.976190,0,0,10.337280
105552,53500,21/11/2018,2018-11-21,21/11/2019,1974-04-01,2011-10-05,0,1,1,1,...,3.495,830,0.0,44,7,37,0.840909,0,0,8.961879
105553,53501,21/11/2018,2018-11-21,21/11/2019,1946-09-15,1982-02-02,0,1,1,1,...,4.555,1399,0.0,72,36,36,0.500000,0,1,9.717760


In [16]:
dataset[['claims_frequency', 'Value_vehicle_log_transfomed', 'fuel_type_encoded', 'Value_vehicle', 'N_doors', 'Length', 'Cylinder_capacity', 'Power', ]].corr(method='spearman')

,claims_frequency,Value_vehicle_log_transfomed,fuel_type_encoded,Value_vehicle,N_doors,Length,Cylinder_capacity,Power
claims_frequency,1.000000,0.030932,0.029268,0.030932,0.020564,0.015729,0.030638,0.031940
Value_vehicle_log_transfomed,0.030932,1.000000,0.473941,1.000000,0.240891,0.766530,0.693616,0.763393
fuel_type_encoded,0.029268,0.473941,1.000000,0.473941,0.283055,0.310847,0.565008,0.306028
Value_vehicle,0.030932,1.000000,0.473941,1.000000,0.240891,0.766530,0.693616,0.763393
N_doors,0.020564,0.240891,0.283055,0.240891,1.000000,0.010196,0.171201,0.219939
Length,0.015729,0.766530,0.310847,0.766530,0.010196,1.000000,0.637831,0.594449
Cylinder_capacity,0.030638,0.693616,0.565008,0.693616,0.171201,0.637831,1.000000,0.579250
Power,0.031940,0.763393,0.306028,0.763393,0.219939,0.594449,0.579250,1.000000


### Summary: All Engineered Features vs Claims Frequency

In [18]:
engineered_features = [
    'driver_age_at_contract_start',
    'driver_experience_age',
    'driver_age_experience_age_diff',
    'driver_age_experience_ratio_proxy_for_driving_experience',
    'lapse_flag',
    'Payment',
    'R_Claims_history',
    'fuel_type_encoded',
    'Value_vehicle_log_transfomed',
    'N_doors',
    'Length',
    'Cylinder_capacity',
    'Power'
]

dataset[['claims_frequency'] + engineered_features].corr(method='spearman')[['claims_frequency']].sort_values(by='claims_frequency', ascending=False)


,claims_frequency
claims_frequency,1.000000
R_Claims_history,0.267851
driver_age_experience_ratio_proxy_for_driving_experience,0.045684
Payment,0.044942
Power,0.031940
Value_vehicle_log_transfomed,0.030932
Cylinder_capacity,0.030638
fuel_type_encoded,0.029268
N_doors,0.020564
Length,0.015729
